In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

HERE = Path.cwd()

SEASON_CSV = HERE / "season_upset_frequency_summary_all_seeds.csv"
LEAGUE_CSV = HERE / "league_upset_frequency_summary_all_seeds.csv"

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [8]:
season_df = pd.read_csv(SEASON_CSV)
league_df = pd.read_csv(LEAGUE_CSV)

season_df["league"] = season_df["league"].str.lower()
league_df["league"] = league_df["league"].str.lower()
season_df["season"] = season_df["season"].astype(int)

ORDER = ["bundesliga", "la_liga", "premier_league", "serie_a"]
NAME = {
    "bundesliga": "bundesliga",
    "la_liga": "la_liga",
    "premier_league": "premier_league",
    "serie_a": "serie_a",
}
COL = {
    "bundesliga": "green",
    "la_liga": "red",
    "premier_league": "orange",
    "serie_a": "blue",
}

# Plot type 1: Per-league time series (one chart per league)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for lg in ORDER:
    sub = season_df[season_df["league"] == lg].copy()
    sub = sub.sort_values("season")
    if sub.empty:
        continue
    ax.plot(
        sub["season"], sub["upset_frequency_avg"],
        marker="o", linewidth=2, alpha=0.9,
        color=COL[lg], label=NAME[lg]
    )

ax.set_title("Upset Frequency by Season (Standings) — Average across 10 simulation seeds")
ax.set_xlabel("Season")
ax.set_ylabel("Upset Frequency")
ax.set_ylim(0.30, 0.50)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.legend(title="League", frameon=True)

out_png = HERE / "upset_frequency_by_season_all_leagues_avg.png"
fig.tight_layout()
fig.savefig(out_png, dpi=200)
plt.close(fig)
print("saved ->", out_png)

saved -> /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/upset_frequency/pure_luck_goals_based_updated/upset_frequency_by_season_all_leagues_avg.png


# Plot type 2: Boxplot of season distributions per league (single chart)

In [13]:
data = [season_df.loc[season_df["league"] == lg, "upset_frequency_avg"].dropna().values
        for lg in ORDER]
labels = [NAME[lg] for lg in ORDER]
colors = [COL[lg] for lg in ORDER]

fig, ax = plt.subplots(figsize=(10, 5))

bp = ax.boxplot(data, labels=labels, showfliers=False, patch_artist=True)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)

ax.set_title("Upset Frequency Distribution Across Leagues (Standings) — Average across 10 simulation seeds")
ax.set_ylabel("Upset Frequency")
ax.set_ylim(0.30, 0.50) 
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

means = [np.nanmean(d) if len(d) else np.nan for d in data]
for i, m in enumerate(means, 1):
    if np.isfinite(m):
        ax.text(i, m + 0.005, f"{m:.3f}", ha="center", va="bottom", fontsize=10, color="black")

out_png = HERE / "upset_frequency_boxplot_by_league_avg.png"
fig.tight_layout()
fig.savefig(out_png, dpi=200)
plt.close(fig)
print("saved ->", out_png)

saved -> /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/upset_frequency/pure_luck_goals_based_updated/upset_frequency_boxplot_by_league_avg.png


# Plot type 3: League totals (bar chart of league-level averages)

In [15]:
league_sorted = league_df.copy()
league_sorted["league"] = pd.Categorical(league_sorted["league"], categories=ORDER, ordered=True)
league_sorted = league_sorted.sort_values("league")

x_labels = [NAME[lg] for lg in league_sorted["league"].astype(str)]
vals = league_sorted["upset_frequency_avg"].to_numpy()
bar_colors = [COL[lg] for lg in league_sorted["league"].astype(str)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x_labels, vals, color=bar_colors, alpha=0.9)

ax.set_title("Average Upset Frequency by League (Standings) — Average across 10 simulation seeds")
ax.set_ylabel("Upset Frequency")
ax.set_ylim(0.00, 0.50)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

for i, v in enumerate(vals):
    ax.text(i, min(v + 0.01, 0.49), f"{v:.3f}", ha="center", va="bottom", fontsize=10, color="black")

out_png = HERE / "upset_frequency_by_league_totals_avg.png"
fig.tight_layout()
fig.savefig(out_png, dpi=200)
plt.close(fig)
print("saved ->", out_png)

saved -> /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/upset_frequency/pure_luck_goals_based_updated/upset_frequency_by_league_totals_avg.png
